In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "manual")          # the job will pass its run ID here
RUN_ID = dbutils.widgets.get("run_id")

SILVER, GOLD = "workspace.climate_silver", "workspace.climate_gold"
EXPECTED_DISTRICTS = 290
LATEST_EXPECTED = "2025-12-31"   # update once monthly incremental loads start

results = []

def check(name, value, passed, detail=""):
    results.append((RUN_ID, name, str(value), bool(passed), detail))
    print(f"{'PASS' if passed else 'FAIL'}  {name}: {value}  {detail}")

# --- Silver weather ---
w = spark.table(f"{SILVER}.weather_daily")
n_w, n_q = w.count(), spark.table(f"{SILVER}.weather_quarantine").count()
share = n_q / (n_w + n_q)
check("quarantine_share", f"{share:.5%}", share < 0.001, "must be < 0.1%")

latest = str(w.agg(F.max("date")).first()[0])
check("weather_latest_date", latest, latest >= LATEST_EXPECTED, f"expected >= {LATEST_EXPECTED}")

n_d = w.select("district_id").distinct().count()
check("weather_districts", n_d, n_d == EXPECTED_DISTRICTS)

# --- Silver reference tables ---
for t in ["districts", "district_soil", "district_population"]:
    n = spark.table(f"{SILVER}.{t}").count()
    check(f"{t}_rows", n, n == EXPECTED_DISTRICTS)

# --- Gold risk score ---
latest_scores = spark.table(f"{GOLD}.district_risk_latest")
n_latest = latest_scores.count()
check("latest_scores_complete", n_latest, n_latest == EXPECTED_DISTRICTS)

bad = latest_scores.filter("risk_score < 0 OR risk_score > 100 OR risk_tier IS NULL").count()
check("scores_valid", bad, bad == 0, "rows outside 0-100 or missing tier")

# --- Gold allocation reconciles with budget ---
alloc = spark.table(f"{GOLD}.resource_allocation")
if alloc.count() > 0:
    s = alloc.agg(F.sum("allocated_kes").alias("allocated"),
                  F.max("total_budget_kes").alias("budget"),
                  F.count("*").alias("rows")).first()
    diff = abs(s.allocated - s.budget)
    check("allocation_reconciles", f"{diff:,.0f} KES off", diff <= s.rows, "within rounding")
else:
    check("allocation_reconciles", "nothing eligible", True, "no relief triggered this season")

# --- Record results, then fail loudly if anything failed ---
(spark.createDataFrame(results, "run_id string, check_name string, value string, passed boolean, detail string")
      .withColumn("checked_at", F.current_timestamp())
      .write.mode("append").saveAsTable(f"{GOLD}.pipeline_runs"))

failed = [row[1] for row in results if not row[3]]
if failed:
    raise Exception(f"Data quality checks failed: {failed}")
print("\nAll checks passed")